# C-03 · Particle Data and Velocity Distribution Functions

Menura saves particle snapshots every `rate_save_particles_cst` iterations (default 4000).  Each file contains position and velocity for every active macro-particle on one MPI rank.

**File format:** NumPy `.npy` arrays with shape `[N_active, 7]` in 3D:
```
column 0: rx   (x-position, normalised to d_i)
column 1: ry
column 2: rz
column 3: vx   (velocity, normalised to Alfvén speed v_A)
column 4: vy
column 5: vz
column 6: ID   (species: 0 = solar wind, 1 = cometary/planetary)
```

For large runs the file is split into chunks of 100 M particles; this notebook handles the chunked case.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, glob

# --- Configure ---
RUN_DIR  = '../menura'
PRODUCTS = os.path.join(RUN_DIR, 'products')

# Read parameters
p = np.recfromtxt(os.path.join(PRODUCTS, 'parameters.txt'))
params = {t[0].decode('UTF-8'): float(t[1]) for t in p}

DX         = params['dX']
DT         = params['dt']
NB_PROC_Y  = int(params.get('mpi_nb_proc_y', params.get('mpi_nb_proc', 1)))
NB_PROC_Z  = int(params.get('mpi_nb_proc_z', 1))
RATE_SAVE_PA = int(params['rate_save_particles_cst'])
LEN_X      = int(params['len_x_cst'])
LEN_Y      = int(params['len_y_cst'])

# Alfvén speed (for velocity units)
e_ch = 1.602e-19; m_i = 1.673e-27; mu0 = 1.257e-6
n0_SI = params.get('n0_SI', 3e6)
B0_SI = params.get('B0_SI', 3e-9)
v_A   = B0_SI / np.sqrt(mu0 * m_i * n0_SI)   # m/s

print(f'Alfvén speed: v_A = {v_A/1e3:.1f} km/s')
print(f'Particles saved every {RATE_SAVE_PA} iterations')

## 1. Load particle data from all ranks

The file naming follows: `particles_FILE_it{NNN}_rank_{Y}_{Z}.npy`

If the particle count exceeds 100 M on a rank, multiple files are written with `_FILE_` replaced by sequential chunk numbers.

In [ ]:
def load_particles(iteration, products_dir, nb_y, nb_z):
    """
    Load all particle data for a given iteration, assembled from all ranks.
    Handles chunked files (multiple files per rank).
    
    Returns
    -------
    np.ndarray, shape (N_total, 7)
    """
    all_parts = []
    for ry in range(nb_y):
        for rz in range(nb_z):
            # Find all chunk files for this rank/iteration
            pattern = os.path.join(products_dir,
                                   f'particles_*_it{iteration}_rank_{ry}_{rz}.npy')
            chunk_files = sorted(glob.glob(pattern))
            if not chunk_files:
                print(f'  Warning: no particle files found for rank ({ry},{rz}) it={iteration}')
                continue
            for fn in chunk_files:
                chunk = np.load(fn)
                all_parts.append(chunk)
    if not all_parts:
        raise FileNotFoundError(f'No particle files found for it={iteration}')
    return np.concatenate(all_parts, axis=0)

# Load particles at the first (and possibly only) saved iteration
pa = load_particles(RATE_SAVE_PA, PRODUCTS, NB_PROC_Y, NB_PROC_Z)

rx, ry_p, rz = pa[:, 0], pa[:, 1], pa[:, 2]
vx, vy, vz   = pa[:, 3], pa[:, 4], pa[:, 5]
species      = pa[:, 6].astype(int)

print(f'Total particles loaded: {len(pa):,}')
print(f'  Species 0 (SW):   {(species==0).sum():,}')
print(f'  Species 1 (pla/com): {(species==1).sum():,}')

## 2. Spatial distribution of particles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, sp, label, color in zip(axes, [0, 1],
                                 ['Solar wind', 'Secondary'],
                                 ['steelblue', 'tomato']):
    mask = (species == sp)
    # 2D histogram in x–y midplane (take slice around z midpoint)
    iz_mid = LEN_Y // 2   # approximate midplane
    z_mid  = iz_mid * DX
    z_range = DX * 5
    z_mask = mask & (np.abs(rz - z_mid) < z_range)
    
    h, xedges, yedges = np.histogram2d(
        rx[z_mask], ry_p[z_mask],
        bins=[LEN_X//2, LEN_Y//2],
        range=[[0, LEN_X*DX], [0, LEN_Y*DX]]
    )
    im = ax.pcolormesh(xedges, yedges, h.T, cmap='plasma' if sp==1 else 'Blues')
    plt.colorbar(im, ax=ax, label='Particle count')
    ax.set_xlabel('x [d_i]')
    ax.set_ylabel('y [d_i]')
    ax.set_title(f'{label} ions — x–y projection')

plt.tight_layout()
plt.savefig('particle_spatial.png', dpi=150)
plt.show()

## 3. Velocity distribution function (VDF)

The 1-D VDF along each velocity component tells us whether the distribution is Maxwellian, beam-like, or shows suprathermal tails.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
v_range = 5.0   # normalised velocity range to plot
bins    = 100

for sp, label, color in [(0, 'Solar wind', 'steelblue'), (1, 'Secondary', 'tomato')]:
    mask = (species == sp)
    for ax, v, vlabel in zip(axes, [vx[mask], vy[mask], vz[mask]], ['vx', 'vy', 'vz']):
        counts, edges = np.histogram(v, bins=bins,
                                     range=(-v_range, v_range), density=True)
        centers = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(centers, counts, label=label, color=color)

for ax, vlabel in zip(axes, ['vx', 'vy', 'vz']):
    ax.set_xlabel(f'{vlabel} [v_A]')
    ax.set_ylabel('f(v)  [normalised]')
    ax.set_title(f'VDF — {vlabel}')
    ax.set_yscale('log')
    ax.legend()

plt.tight_layout()
plt.savefig('vdf_1d.png', dpi=150)
plt.show()

## 4. 2-D VDF (vx–vy plane)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, sp, label in [(axes[0], 0, 'Solar wind'), (axes[1], 1, 'Secondary')]:
    mask = (species == sp)
    h, xedges, yedges = np.histogram2d(
        vx[mask], vy[mask],
        bins=80,
        range=[[-v_range, v_range], [-v_range, v_range]]
    )
    h_log = np.log10(h + 1)   # log scale for dynamic range
    im = ax.pcolormesh(xedges, yedges, h_log.T, cmap='inferno')
    plt.colorbar(im, ax=ax, label='log10(count + 1)')
    ax.set_xlabel('vx [v_A]')
    ax.set_ylabel('vy [v_A]')
    ax.set_title(f'{label} VDF (vx–vy)')
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('vdf_2d.png', dpi=150)
plt.show()

## 5. Relating parameters to physical units

Velocities in the particle files are normalised to the Alfvén speed `v_A`.  Positions are normalised to the ion inertial length `d_i`.

In [ ]:
omega_ci = e_ch * B0_SI / m_i      # ion cyclotron frequency [rad/s]
d_i      = v_A / omega_ci           # ion inertial length [m]

print('Physical scales:')
print(f'  v_A   = {v_A/1e3:.2f} km/s')
print(f'  d_i   = {d_i/1e3:.2f} km')
print(f'  1/Ω_ci = {1/omega_ci:.4f} s')
print()
print('To convert particle velocities to km/s: multiply by v_A/1e3')
print('To convert positions to km: multiply by d_i/1e3')

# Example: thermal speed of solar wind ions
sw_mask = (species == 0)
v_th_sim = np.std(vx[sw_mask])   # in units of v_A
v_th_kms = v_th_sim * v_A / 1e3
print(f'\nSW ion thermal speed: {v_th_sim:.3f} v_A = {v_th_kms:.1f} km/s')

## 6. Phase-space density as a function of distance from the obstacle

Select secondary ions within a spherical shell around the obstacle and compute their VDF.

In [ ]:
# Obstacle centre (from parameters.h defaults: 0.5 × domain length)
cx = params.get('centre_x', 0.5) * LEN_X * DX
cy = params.get('centre_y', 0.5) * LEN_Y * DX
cz = params.get('centre_z', 0.5) * LEN_Y * DX   # assume cube-ish

# Radial distance from obstacle centre [d_i]
r = np.sqrt((rx - cx)**2 + (ry_p - cy)**2 + (rz - cz)**2)

# Select secondary ions in shell 30–60 d_i from obstacle
r_inner, r_outer = 30.0, 60.0
shell_mask = (species == 1) & (r > r_inner) & (r < r_outer)

print(f'Secondary ions in shell {r_inner}–{r_outer} d_i: {shell_mask.sum():,}')

if shell_mask.sum() > 100:
    fig, ax = plt.subplots(figsize=(6, 4))
    v_tot = np.sqrt(vx[shell_mask]**2 + vy[shell_mask]**2 + vz[shell_mask]**2)
    ax.hist(v_tot, bins=60, density=True, color='tomato', alpha=0.8)
    ax.set_xlabel('|v| [v_A]')
    ax.set_ylabel('Normalised count')
    ax.set_title(f'Speed distribution of secondary ions\n({r_inner}–{r_outer} d_i from obstacle)')
    plt.tight_layout()
    plt.savefig('shell_vdf.png', dpi=150)
    plt.show()
else:
    print('Too few particles in shell for a meaningful VDF.')

## 7. Exercises

1. In `parameters.h`, `pool_size_cst` is computed as `1.2 × nb_part_node_cst × len_x × len_y × len_z`.  Compare this to the actual number of particles loaded.  What fraction of the pool is occupied?

2. The solar wind drift velocity in the x-direction should be close to `v0` (set in `simu_param`; default 0, i.e., plasma at rest in normalised frame).  Verify this by computing `np.mean(vx[species==0])`.

3. Secondary (cometary/planetary) ions are injected with a source distribution.  Based on the VDF you plotted, do they look thermalized, beam-like, or something in between?  What would full thermalization look like?

4. The particle files can be very large for 3D runs.  How many gigabytes would a single particle snapshot occupy for the default parameters (`pool_size_cst ≈ 1.2 × 40 × 400 × 100 × 100 = 192 M` particles, each stored as 7 float32)?  Calculate it.

5. Modify the spatial histogram to use `pcolormesh` with logarithmic color scaling.  Which region shows the lowest particle density?  Why?